# Class 18: Better Than Guessing
### Linear Regression and the Coefficient of Determination

**Record your answers on the paper handout, not here.**

In [ ]:
import numpy as np
from datascience import *
import matplotlib.pyplot as plt
%matplotlib inline

print("Ready!")

### Helper functions

The same four we have been building all along. `slope` and `intercept` come straight from $r$ — that is the whole regression line.

In [ ]:
def standard_units(any_numbers):
    "Convert any array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

---
## Part 1. Two Baselines

> **Handout Q1.1 and Q1.2** — answer both before running anything below.

---
## Part 2. How Good Is the Cricket Line?

As temperature rises, crickets chirp faster. Fifteen observations.

In [ ]:
cricket = Table.read_table("./data/cricket_thermometer.csv")
cricket

### Step 1 — The mean baseline

If you had to predict every temperature with a single number, this horizontal line is the best you could do.

In [ ]:
chirps = cricket.column('Chirps_per_sec')
temp   = cricket.column('Temperature_deg_F')

temp_mean = np.mean(temp)
print(f"Mean temperature: {temp_mean:.2f} F")

plt.figure(figsize=(8, 5))
plt.scatter(chirps, temp, color='steelblue', s=60, zorder=5, label='Observations')
plt.axhline(y=temp_mean, color='tomato', linewidth=2, linestyle='--',
            label=f'Mean = {temp_mean:.1f} F')
for xi, yi in zip(chirps, temp):
    plt.plot([xi, xi], [yi, temp_mean], color='tomato', alpha=0.4, linewidth=1.5)
plt.xlabel('Chirps per Second'); plt.ylabel('Temperature (F)')
plt.title('Error Before Regression')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

Each red segment is the error you would make at that point. $SS_{\text{Total}}$ is the sum of those segments **squared**.

$$SS_{\text{Total}} = \sum (y_i - \bar{y})^2$$

In [ ]:
deviations = temp - temp_mean
SS_total   = np.sum(... ** 2)      # square the deviations, then add them up

print(f"SS_Total = {SS_total:.2f}")

> **Handout Q2.1**

### Step 2 — Fit the line

In [ ]:
m = slope(cricket, 'Chirps_per_sec', 'Temperature_deg_F')
b = intercept(cricket, 'Chirps_per_sec', 'Temperature_deg_F')

print(f"Slope:     {m:.4f}")
print(f"Intercept: {b:.4f}")
print(f"Temperature = {m:.2f} * Chirps_per_sec + {b:.2f}")

> **Handout Q2.2**

### Step 3 — The error that remains

$$SS_{\text{Residual}} = \sum (y_i - \hat{y}_i)^2$$

In [ ]:
predicted = m * chirps + b
residuals = temp - ...              # subtract the PREDICTED temperature, not the mean

plt.figure(figsize=(8, 5))
plt.scatter(chirps, temp, color='steelblue', s=60, zorder=5, label='Observations')
plt.plot(chirps, predicted, color='seagreen', linewidth=2, label='Regression line')
plt.axhline(y=temp_mean, color='tomato', linewidth=2, linestyle='--',
            label=f'Mean = {temp_mean:.1f} F', alpha=0.5)
for xi, yi, yh in zip(chirps, temp, predicted):
    plt.plot([xi, xi], [yi, yh], color='seagreen', alpha=0.5, linewidth=1.5)
plt.xlabel('Chirps per Second'); plt.ylabel('Temperature (F)')
plt.title('Error After Regression')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

In [ ]:
SS_residual = np.sum(residuals ** 2)
print(f"SS_Total    = {SS_total:.2f}")
print(f"SS_Residual = {SS_residual:.2f}")

### Step 4 — Compute $R^2$

$$R^2 = 1 - \frac{SS_{\text{Residual}}}{SS_{\text{Total}}}$$

In [ ]:
R_squared = 1 - (... / ...)

r = correlation(cricket, 'Chirps_per_sec', 'Temperature_deg_F')

print(f"R^2 (from sums of squares) = {R_squared:.4f}")
print(f"r                          = {r:.4f}")
print(f"r^2                        = {r**2:.4f}")
print()
print("Do R^2 and r^2 match?", np.isclose(R_squared, r**2))

For simple linear regression these are the *same quantity*. Squaring the correlation gives you the fraction of the variation in $y$ that a straight line in $x$ accounts for.

> **Handout Q2.3 and Q2.4**

---
## Part 3. Where the Line Stops Working

In [ ]:
for c_ in [19, 40, 0]:
    print(f"{c_:5.1f} chirps/sec  ->  {m * c_ + b:6.1f} F")

print()
print(f"Observed chirp rates ran from {chirps.min():.1f} to {chirps.max():.1f}")

> **Handout Q3.1**

### A line that is worse than useless

Keep the fitted slope, but force the line through the origin.

In [ ]:
predicted_bad = m * chirps + 0
SS_residual_bad = np.sum((temp - predicted_bad) ** 2)
R2_bad = 1 - (SS_residual_bad / SS_total)

print(f"At 15 chirps/sec this line predicts {m*15:6.1f} F")
print(f"Actual temperatures ran from {temp.min():.1f} to {temp.max():.1f} F")
print()
print(f"SS_Residual = {SS_residual_bad:.1f}   (compare to {SS_residual:.1f} for the real line)")
print(f"R^2         = {R2_bad:.3f}")

> **Handout Q3.2**

---
## Part 4. A Better Number That Is a Worse Model

A different data set. Seven points, and we fit a straight line to them exactly the way we just did.

In [ ]:
x_vals = np.arange(0, 7)
y_vals = x_vals ** 2

data = Table().with_columns('x', x_vals, 'y', y_vals)

m2 = slope(data, 'x', 'y')
b2 = intercept(data, 'x', 'y')
pred2 = m2 * x_vals + b2

SS_total2    = np.sum((y_vals - np.mean(y_vals)) ** 2)
SS_residual2 = np.sum((y_vals - pred2) ** 2)
R2_second    = 1 - SS_residual2 / SS_total2

print(f"Line: y = {m2:.1f}x + ({b2:.1f})")
print(f"SS_Total = {SS_total2:.0f}   SS_Residual = {SS_residual2:.0f}")
print()
print(f"R^2, second data set = {R2_second:.4f}")
print(f"R^2, crickets        = {R_squared:.4f}")

> **Handout Q4.1** — look at the plot below before you answer.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(x_vals, y_vals, color='steelblue', s=80, zorder=5, label='Data')
plt.plot(x_vals, pred2, color='seagreen', linewidth=2, label=f'y = {m2:.0f}x + ({b2:.0f})')
for xi, yi, yh in zip(x_vals, y_vals, pred2):
    plt.plot([xi, xi], [yi, yh], color='seagreen', alpha=0.5, linewidth=1.5)
plt.xlabel('x'); plt.ylabel('y')
plt.title(f'Second Data Set   (R^2 = {R2_second:.3f})')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

### The residual plot

Plot the leftover error against x. If the line is the right model, the residuals should scatter with no pattern — the line has already used up everything it can.

In [ ]:
residuals2 = y_vals - pred2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(chirps, residuals, color='steelblue', s=60)
axes[0].axhline(0, color='k', linewidth=1)
axes[0].set_xlabel('Chirps per Second'); axes[0].set_ylabel('Residual (F)')
axes[0].set_title(f'Crickets   (R^2 = {R_squared:.3f})')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(x_vals, residuals2, color='steelblue', s=80)
axes[1].axhline(0, color='k', linewidth=1)
axes[1].set_xlabel('x'); axes[1].set_ylabel('Residual')
axes[1].set_title(f'Second data set   (R^2 = {R2_second:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

> **Handout Q4.2 and Q4.3**